# Synthetic Noisy Time Series Dataset Demo

This demo notebook loads the synthetic noisy time series dataset and evaluates moving average forecasting performance relative to naive last-value forecasting.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

In [ ]:
import json
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt

## Data Loading Helper
We use the standard GitHub data loading pattern with local fallback.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-29c492-empirical-audit-of-moving-average-baseli/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

data_payload = load_data()
print("Loaded data successfully!")

## Configuration
Define parameters for moving average window size and evaluation.

In [ ]:
# Tunable parameters
WINDOW_SIZE = 3
NUM_TRIALS_TO_PROCESS = 3

## Processing Trials
Compute moving average and naive last-value forecasts for each trial and evaluate errors.

In [ ]:
datasets = data_payload.get("datasets", [])
examples = datasets[0].get("examples", []) if datasets else []

results = []
for i, ex in enumerate(examples[:NUM_TRIALS_TO_PROCESS]):
    series = json.loads(ex["input"])
    true_mean = float(ex["output"])
    trial_id = ex["metadata_trial_id"]
    length = ex["metadata_length"]
    noise_var = ex["metadata_noise_variance"]
    
    # Simple moving average forecast (mean of last WINDOW_SIZE elements)
    if len(series) >= WINDOW_SIZE:
        ma_forecast = np.mean(series[-WINDOW_SIZE:])
    else:
        ma_forecast = np.mean(series)
        
    # Naive last-value forecast
    naive_forecast = series[-1]
    
    ma_error = abs(ma_forecast - true_mean)
    naive_error = abs(naive_forecast - true_mean)
    
    results.append({
        "trial_id": trial_id,
        "series": series,
        "true_mean": true_mean,
        "ma_forecast": ma_forecast,
        "naive_forecast": naive_forecast,
        "ma_error": ma_error,
        "naive_error": naive_error
    })
    print(f"Trial {trial_id}: MA Error={ma_error:.4f}, Naive Error={naive_error:.4f}")

## Results and Visualization
Plot the time series and forecasts for the trials.

In [ ]:
plt.figure(figsize=(10, 4))
for res in results:
    plt.plot(res["series"], marker='o', label=f"Trial {res['trial_id']}")
plt.title("Synthetic Noisy Time Series Trials")
plt.xlabel("Time Step")
plt.ylabel("Value")
plt.legend()
plt.grid(True)
plt.show()